# PROJECT 4: DATA FITTING AND ITS APPLICATIONS

## I. STUDENT INFORMATION
* **Full Name:** Nguyễn Nhựt Huy
* **Student ID (MSSV):** 24127398
* **Class:** 24C11
* **Course:** Applied Mathematics and Statistics

## II. THEORY OVERVIEW

---

### 1. Linear Regression Model

* **Definition:** Given a dataset of $n$ observations $\{(x_i, y_i)\}_{i=1}^{n}$, where $x_i = (x_{i1}, x_{i2}, \dots, x_{ip})^{\top}$ is the vector of $p$ predictor variables and $y_i$ is the corresponding response, the **multiple linear regression model** assumes:
  $$y_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \dots + \beta_p x_{ip} + \varepsilon_i$$
  where $\beta_0$ is the intercept, $\beta_1, \dots, \beta_p$ are the regression coefficients, and $\varepsilon_i$ is the random error term.

* **Matrix Form:** Let $X$ be the $n \times (p+1)$ **design matrix** whose first column is all ones (for the intercept) and $y$ be the response vector. The model becomes:
  $$y = X \beta + \varepsilon$$

---

### 2. Ordinary Least Squares (OLS)

* **Objective:** Find the coefficient vector $\beta$ that minimizes the **Residual Sum of Squares (RSS)**:
  $$\text{RSS}(\beta) = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \|y - X\beta\|_2^2$$

* **Normal Equations & Closed-form Solution:** Setting the gradient of RSS to zero yields the **normal equations** $X^{\top}X\beta = X^{\top}y$. If $X^{\top}X$ is invertible, the optimal coefficients are:
  $$\hat{\beta} = (X^{\top} X)^{-1} X^{\top} y$$

---

### 3. Model Evaluation Metrics

Let $y$ be the true values and $\hat{y}$ the predicted values, with $\bar{y}$ the mean of $y$.

1. **MAE (Mean Absolute Error):**
   $$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

2. **MSE (Mean Squared Error):**
   $$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \frac{\text{RSS}}{n}$$

3. **RMSE (Root Mean Squared Error):**
   $$\text{RMSE} = \sqrt{\text{MSE}}$$

4. **$R^2$ (Coefficient of Determination):**
   $$R^2 = 1 - \frac{\text{RSS}}{\text{TSS}}, \qquad \text{TSS} = \sum_{i=1}^{n} (y_i - \bar{y})^2$$

5. **Adjusted $R^2$:** Penalizes the number of predictors $p$ to avoid overfitting:
   $$\bar{R}^2 = 1 - (1 - R^2)\frac{n - 1}{n - p - 1}$$

---

### 4. Cross-Validation and Feature Selection

* **$k$-Fold Cross-Validation:** The training data is partitioned into $k$ equal-sized folds. In each iteration, one fold is held out for validation while the model is trained on the remaining $k-1$ folds. The validation score is averaged over all $k$ folds to obtain a more reliable estimate of generalization performance.

* **Feature Selection:** Each feature is evaluated individually by its mean CV $R^2$; features that yield $R^2 > 0$ are retained. This identifies the most predictive features (e.g., `alcohol`) and supports building a reduced model with engineered features.

## III. IMPLEMENTATION

### Import necessary libraries

In [1]:
import math
import pandas as pd
from sklearn.model_selection import train_test_split

### Support Functions: Vector & Matrix Helpers

In [2]:
def iszero(number, eps=1e-9):
    return abs(number) < eps

def is_zero_vector(v):
    return all(iszero(x) for x in v)

def mul_vector(v, alpha):
    return [x * alpha for x in v]

def add_vector(u, v):
    return [u[i] + v[i] for i in range(len(u))]

def subtrac_vector(u, v):
    return [u[i] - v[i] for i in range(len(u))]

def inner(u, v):
    return sum(u[i] * v[i] for i in range(len(u)))

def norm(v):
    return math.sqrt(inner(v, v))

def identity_matrix(n):
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]

In [3]:
def transpose(M):
    return [[M[j][i] for j in range(len(M))] for i in range(len(M[0]))]

def multiply_matrices(A, B):
    rows_A, cols_A = len(A), len(A[0])
    rows_B, cols_B = len(B), len(B[0])
    if cols_A != rows_B:
        raise ValueError("Invalid matrix dimensions for multiplication!")

    C = [[0.0 for _ in range(cols_B)] for _ in range(rows_A)]
    for i in range(rows_A):
        for j in range(cols_B):
            C[i][j] = sum(A[i][k] * B[k][j] for k in range(cols_A))
    return C

def invert_matrix(A):
    n = len(A)
    I = identity_matrix(n)
    aug = [A[i][:] + I[i][:] for i in range(n)]

    for col in range(n):
        max_row = col
        for r in range(col + 1, n):
            if abs(aug[r][col]) > abs(aug[max_row][col]):
                max_row = r
        aug[col], aug[max_row] = aug[max_row], aug[col]

        pivot = aug[col][col]
        if iszero(pivot):
            raise ValueError("Matrix is singular, cannot invert!")

        aug[col] = [x / pivot for x in aug[col]]

        for r in range(n):
            if r != col:
                factor = aug[r][col]
                aug[r] = [aug[r][j] - factor * aug[col][j] for j in range(2 * n)]

    return [row[n:] for row in aug]

### Ordinary Least Squares Solver

In [4]:
def calculate_beta_ols(X_list, y_list):
    X_T = transpose(X_list)
    X_T_X = multiply_matrices(X_T, X_list)
    inv_X_T_X = invert_matrix(X_T_X)
    
    y_mat = [[y] for y in y_list]
    X_T_y = multiply_matrices(X_T, y_mat)
    
    beta_mat = multiply_matrices(inv_X_T_X, X_T_y)
    return [row[0] for row in beta_mat]

In [5]:
def predict_single(x_row, beta):
    return inner(x_row, beta)

def predict_all(X_list, beta):
    return [predict_single(row, beta) for row in X_list]

### Evaluation Metrics

In [6]:
def calculate_mae(y_true, y_pred):
    diff = subtrac_vector(y_true, y_pred)
    return sum(abs(x) for x in diff) / len(y_true)

def calculate_rss(y_true, y_pred):
    diff = subtrac_vector(y_true, y_pred)
    return sum(x ** 2 for x in diff)

def calculate_tss(y_true):
    y_mean = sum(y_true) / len(y_true)
    return sum((y - y_mean) ** 2 for y in y_true)

def calculate_mse(y_true, y_pred):
    return calculate_rss(y_true, y_pred) / len(y_true)

def calculate_rmse(y_true, y_pred):
    return math.sqrt(calculate_mse(y_true, y_pred))

def calculate_r2(y_true, y_pred):
    rss = calculate_rss(y_true, y_pred)
    tss = calculate_tss(y_true)
    return 1 - (rss / tss)

def calculate_adj_r2(y_true, y_pred, p):
    n = len(y_true)
    r2 = calculate_r2(y_true, y_pred)
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

### Cross-Validation for Feature Selection

In [7]:
def k_fold_split(X_single_col, y_list, k=5):
    n = len(y_list)
    fold_size = n // k
    folds = []

    for fold_idx in range(k):
        val_start = fold_idx * fold_size
        val_end = (fold_idx + 1) * fold_size if fold_idx < k - 1 else n

        X_val_raw = X_single_col[val_start:val_end]
        y_val = y_list[val_start:val_end]

        X_train_raw = X_single_col[:val_start] + X_single_col[val_end:]
        y_train = y_list[:val_start] + y_list[val_end:]

        X_train = [[1.0, val] for val in X_train_raw]
        X_val = [[1.0, val] for val in X_val_raw]

        folds.append((X_train, y_train, X_val, y_val))

    return folds

def evaluate_single_feature_cv(X_single_col, y_list, k=5):
    folds = k_fold_split(X_single_col, y_list, k=k)
    r2_scores = []
    mse_scores = []

    for X_train, y_train, X_val, y_val in folds:
        beta = calculate_beta_ols(X_train, y_train)
        y_pred = predict_all(X_val, beta)
        r2_scores.append(calculate_r2(y_val, y_pred))
        mse_scores.append(calculate_mse(y_val, y_pred))

    mean_r2 = sum(r2_scores) / k
    mean_mse = sum(mse_scores) / k
    return mean_r2, mean_mse

### Feature Engineering

In [8]:
def add_engineered_features(X_raw_list, feature_names_list):
    idx_free = feature_names_list.index("free sulfur dioxide")
    idx_total = feature_names_list.index("total sulfur dioxide")
    idx_fixed = feature_names_list.index("fixed acidity")
    idx_volatile = feature_names_list.index("volatile acidity")
    idx_ph = feature_names_list.index("pH")
    idx_alcohol = feature_names_list.index("alcohol")

    X_new = []
    for row in X_raw_list:
        new_row = row[:]
        
        bound_so2_ratio = (row[idx_total] - row[idx_free]) / (row[idx_total] + 1e-5)
        acid_ratio = row[idx_fixed] / (row[idx_volatile] + 1e-5)
        alc_volatile_ratio = row[idx_alcohol] / (row[idx_volatile] + 1e-5)

        new_row.extend([bound_so2_ratio, acid_ratio, alc_volatile_ratio])
        X_new.append(new_row)

    new_feature_names = feature_names_list + ["bound_so2_ratio", "acid_ratio", "alc_volatile_ratio"]
    return X_new, new_feature_names

## IV. TEST CASES

### Question A: Multiple Linear Regression on Wine Dataset

In [9]:
df = pd.read_csv("wine.csv")
df.columns = df.columns.str.strip().str.replace('"', '')

X = df.drop(columns="quality")
y = df["quality"]

X_train_df, X_test_df, y_train_df, y_test_df = train_test_split(
    X, y, test_size=0.2, random_state=42
)

feature_names = list(X_train_df.columns)
target_name = y_train_df.name

X_train_raw = X_train_df.values.tolist()
X_test_raw = X_test_df.values.tolist()
y_train = y_train_df.tolist()
y_test = y_test_df.tolist()

X_train = [[1.0] + row for row in X_train_raw]
X_test = [[1.0] + row for row in X_test_raw]

beta = calculate_beta_ols(X_train, y_train)

print("=== BETA COEFFICIENTS (COMPUTED ON TRAIN SET) ===")
print(f"Beta_0 (Intercept) = {beta[0]:.6f}")
for idx, name in enumerate(feature_names, start=1):
    print(f"Beta_{idx} ({name}) = {beta[idx]:.6f}")

y_pred_test = predict_all(X_test, beta)

mse_val = calculate_mse(y_test, y_pred_test)
rmse_val = calculate_rmse(y_test, y_pred_test)
mae_val = calculate_mae(y_test, y_pred_test)
r2_val = calculate_r2(y_test, y_pred_test)
adj_r2_val = calculate_adj_r2(y_test, y_pred_test, p=len(feature_names))

print("\n=== MODEL EVALUATION ON TEST SET ===")
print(f"MSE  = {mse_val:.6f}")
print(f"RMSE = {rmse_val:.6f}")
print(f"MAE  = {mae_val:.6f}")
print(f"R^2  = {r2_val:.6f}")
print(f"Adjusted R^2 = {adj_r2_val:.6f}")

=== BETA COEFFICIENTS (COMPUTED ON TRAIN SET) ===
Beta_0 (Intercept) = 46.017877
Beta_1 (fixed acidity) = 0.049136
Beta_2 (volatile acidity) = -1.047212
Beta_3 (citric acid) = -0.268870
Beta_4 (residual sugar) = 0.050456
Beta_5 (chlorides) = -1.491027
Beta_6 (free sulfur dioxide) = 0.004691
Beta_7 (total sulfur dioxide) = -0.004282
Beta_8 (density) = -42.709832
Beta_9 (pH) = -0.215565
Beta_10 (sulphates) = 0.747625
Beta_11 (alcohol) = 0.266749

=== MODEL EVALUATION ON TEST SET ===
MSE  = 0.399423
RMSE = 0.631999
MAE  = 0.495979
R^2  = 0.375750
Adjusted R^2 = 0.345632


### Question B: Finding the Strongest Feature via 5-Fold Cross-Validation

In [10]:
X_cols_train = [[row[col_idx] for row in X_train_raw] for col_idx in range(len(feature_names))]

cv_results = []

print("=== EVALUATING 11 FEATURES USING 5-FOLD CROSS VALIDATION ===")
for idx, name in enumerate(feature_names):
    single_col_data = X_cols_train[idx]
    
    mean_r2, mean_mse = evaluate_single_feature_cv(single_col_data, y_train, k=5)
    
    cv_results.append({
        "feature": name,
        "index": idx,
        "mean_r2": mean_r2,
        "mean_mse": mean_mse
    })
    print(f"Feature: {name:<22} | CV R^2: {mean_r2:.6f} | CV MSE: {mean_mse:.6f}")

best_feature_info = max(cv_results, key=lambda item: item["mean_r2"])

print("\n" + "="*50)
print(f"STRONGEST FEATURE: '{best_feature_info['feature']}'")
print(f"With mean CV R^2 = {best_feature_info['mean_r2']:.6f}")
print("="*50)

best_idx = best_feature_info["index"]
X_train_best = [[1.0, row[best_idx]] for row in X_train_raw]
X_test_best = [[1.0, row[best_idx]] for row in X_test_raw]

beta_best = calculate_beta_ols(X_train_best, y_train)

y_pred_best_test = predict_all(X_test_best, beta)

print(f"\n=== BEST UNIVARIATE MODEL ('{best_feature_info['feature']}') ON TEST SET ===")
print(f"Beta_0 (Intercept) = {beta_best[0]:.6f}")
print(f"Beta_1 ({best_feature_info['feature']}) = {beta_best[1]:.6f}")

print("\n=== EVALUATION ON TEST SET ===")
print(f"MSE  = {calculate_mse(y_test, y_pred_best_test):.6f}")
print(f"RMSE = {calculate_rmse(y_test, y_pred_best_test):.6f}")
print(f"MAE  = {calculate_mae(y_test, y_pred_best_test):.6f}")
print(f"R^2  = {calculate_r2(y_test, y_pred_best_test):.6f}")

=== EVALUATING 11 FEATURES USING 5-FOLD CROSS VALIDATION ===
Feature: fixed acidity          | CV R^2: 0.001609 | CV MSE: 0.650522
Feature: volatile acidity       | CV R^2: 0.130385 | CV MSE: 0.567979
Feature: citric acid            | CV R^2: 0.033038 | CV MSE: 0.630537
Feature: residual sugar         | CV R^2: -0.019337 | CV MSE: 0.663055
Feature: chlorides              | CV R^2: -0.011158 | CV MSE: 0.656954
Feature: free sulfur dioxide    | CV R^2: -0.020139 | CV MSE: 0.662774
Feature: total sulfur dioxide   | CV R^2: 0.026531 | CV MSE: 0.630160
Feature: density                | CV R^2: 0.011050 | CV MSE: 0.641557
Feature: pH                     | CV R^2: -0.015351 | CV MSE: 0.660382
Feature: sulphates              | CV R^2: 0.021284 | CV MSE: 0.635775
Feature: alcohol                | CV R^2: 0.241110 | CV MSE: 0.493569

STRONGEST FEATURE: 'alcohol'
With mean CV R^2 = 0.241110

=== BEST UNIVARIATE MODEL ('alcohol') ON TEST SET ===
Beta_0 (Intercept) = 1.725336
Beta_1 (alcohol) = 0.3

### Question C: Mini Data Science Pipeline (Feature Engineering + Selection)

In [11]:
X_train_eng, new_features = add_engineered_features(X_train_raw, feature_names)
X_test_eng, _ = add_engineered_features(X_test_raw, feature_names)

print(f"Total number of features after Feature Engineering: {len(new_features)}")

selected_indices = []

for idx in range(len(new_features)):
    col_data = [row[idx] for row in X_train_eng]
    mean_r2, _ = evaluate_single_feature_cv(col_data, y_train, k=5)
    
    if mean_r2 > 0:
        selected_indices.append(idx)

selected_feature_names = [new_features[i] for i in selected_indices]
print(f"Selected features ({len(selected_feature_names)}): {selected_feature_names}")

X_train_custom = [[1.0] + [row[i] for i in selected_indices] for row in X_train_eng]
X_test_custom = [[1.0] + [row[i] for i in selected_indices] for row in X_test_eng]

beta_custom = calculate_beta_ols(X_train_custom, y_train)

y_pred_custom = predict_all(X_test_custom, beta_custom)

mse_c = calculate_mse(y_test, y_pred_custom)
rmse_c = calculate_rmse(y_test, y_pred_custom)
mae_c = calculate_mae(y_test, y_pred_custom)
r2_c = calculate_r2(y_test, y_pred_custom)
adj_r2_c = calculate_adj_r2(y_test, y_pred_custom, p=len(selected_indices))

print("\n=== RESULTS OF CUSTOM MODEL (QUESTION C) ON TEST SET ===")
print(f"MSE  = {mse_c:.6f}")
print(f"RMSE = {rmse_c:.6f}")
print(f"MAE  = {mae_c:.6f}")
print(f"R^2  = {r2_c:.6f}")
print(f"Adjusted R^2 = {adj_r2_c:.6f}")

Total number of features after Feature Engineering: 14
Selected features (10): ['fixed acidity', 'volatile acidity', 'citric acid', 'total sulfur dioxide', 'density', 'sulphates', 'alcohol', 'bound_so2_ratio', 'acid_ratio', 'alc_volatile_ratio']

=== RESULTS OF CUSTOM MODEL (QUESTION C) ON TEST SET ===
MSE  = 0.394409
RMSE = 0.628020
MAE  = 0.492098
R^2  = 0.383585
Adjusted R^2 = 0.356668


## V. IMPLEMENTATION IDEA AND FUNCTION DESCRIPTIONS

---

### 1. Code Architecture & Implementation Strategy

#### A. Data Models
* **List-of-Lists Representation:** Matrices are represented as Python lists of lists (each inner list is a row).
* **Pure Python + Math:** Computations use plain Python lists and `math`, avoiding external numerical libraries for the core linear algebra.

#### B. Program Execution Workflow
1. **Load Data:** Read `wine.csv` and normalize column names.
2. **Split Data:** Partition into train/test sets (80/20).
3. **Fit OLS:** Compute $\hat{\beta} = (X^{\top}X)^{-1}X^{\top}y$ with custom matrix routines.
4. **Evaluate:** Compute regression metrics on the test set.
5. **Select Features:** Rank all features by 5-fold CV $R^2$ and pick the best one (Question B).
6. **Engineer + Refine:** Create combined chemical features and re-select by CV $R^2$ (Question C).

---

### 2. Detailed Function Descriptions

#### A. Helper Subroutines

| Function Name & Signature | Description | Input | Return |
| :--- | :--- | :--- | :--- |
| `iszero(number, eps)` | Checks whether a number is approximately zero. | `number` (float), `eps` (float) | `bool` |
| `identity_matrix(n)` | Creates an $n \times n$ identity matrix. | `n` (int) | `list of list` |
| `transpose(M)` | Transposes a matrix. | `M` (list of list) | `list of list` |
| `multiply_matrices(A, B)` | Multiplies two matrices. | `A`, `B` (list of list) | `list of list` |
| `invert_matrix(A)` | Inverts $A$ via Gauss-Jordan on $[A \mid I]$. | `A` (list of list) | `list of list` |

#### B. Core Regression Engine

| Function Name & Signature | Description | Input | Return |
| :--- | :--- | :--- | :--- |
| `calculate_beta_ols(X, y)` | Computes OLS coefficients $(X^{\top}X)^{-1}X^{\top}y$. | `X` (list of list), `y` (list) | `list` |
| `predict_single(x_row, beta)` | Computes $\hat{y}$ for one row. | `x_row` (list), `beta` (list) | `float` |
| `predict_all(X, beta)` | Predicts for all rows. | `X` (list of list), `beta` (list) | `list` |

#### C. Evaluation Metrics

| Function Name & Signature | Description | Input | Return |
| :--- | :--- | :--- | :--- |
| `calculate_mae(y_true, y_pred)` | Mean absolute error. | `y_true`, `y_pred` (list) | `float` |
| `calculate_rss(y_true, y_pred)` | Residual sum of squares. | `y_true`, `y_pred` (list) | `float` |
| `calculate_tss(y_true)` | Total sum of squares. | `y_true` (list) | `float` |
| `calculate_mse(y_true, y_pred)` | Mean squared error. | `y_true`, `y_pred` (list) | `float` |
| `calculate_rmse(y_true, y_pred)` | Root mean squared error. | `y_true`, `y_pred` (list) | `float` |
| `calculate_r2(y_true, y_pred)` | Coefficient of determination $R^2$. | `y_true`, `y_pred` (list) | `float` |
| `calculate_adj_r2(y_true, y_pred, p)` | Adjusted $R^2$. | `y_true`, `y_pred` (list), `p` (int) | `float` |

#### D. Cross-Validation & Feature Engineering

| Function Name & Signature | Description | Input | Return |
| :--- | :--- | :--- | :--- |
| `k_fold_split(X_single_col, y, k)` | Splits data into $k$ train/validation folds. | `X_single_col`, `y` (list), `k` (int) | `list` of tuples |
| `evaluate_single_feature_cv(X_col, y, k)` | Mean CV $R^2$ and MSE for one feature. | `X_col`, `y` (list), `k` (int) | `(mean_r2, mean_mse)` |
| `add_engineered_features(X, feature_names)` | Adds ratio/interaction chemical features. | `X` (list of list), `feature_names` (list) | `(X_new, new_names)` |